# Replikasi DMRDF pada Apache Spark — Dataset Flipkart E-Commerce
## Mata Kuliah: Mahadata / Big Data (IF25-40402) — ITERA
### Jurnal Acuan: Narayanan, S., Samuel, P., & Chacko, M. (2020)
*Improving prediction with enhanced Distributed Memory-based Resilient Dataset Filter.*
Journal of Big Data, 7, Article 13. https://doi.org/10.1186/s40537-020-00292-y

---
**Tujuan:** Mereplikasi secara parsial metode Feature Information Gain (FIG) dan
Distributed Memory-based Resilient Dataset Filter (DMRDF) pada dataset Flipkart
E-Commerce menggunakan Apache Spark 3.5.1 dalam mode lokal.


## 0. Setup Environment

In [ ]:
import os, sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

print(f'Python executable : {sys.executable}')
print(f'Python version    : {sys.version}')

import pyspark
print(f'PySpark version   : {pyspark.__version__}')


## 1. Inisialisasi Apache SparkSession (Local Mode)
SparkSession dikonfigurasi dengan mode lokal `local[*]` yang memanfaatkan seluruh
core CPU yang tersedia. Driver memory dibatasi 2 GB agar eksekusi ramah pada
lingkungan komputer lokal.


In [ ]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName('DMRDF_Replication_ITERA_IF25-40402') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.python.worker.reuse', 'true') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('=' * 60)
print(f'Spark Version       : {spark.version}')
print(f'Spark Master        : {spark.sparkContext.master}')
print(f'Default Parallelism : {spark.sparkContext.defaultParallelism}')
print('SparkSession berhasil diinisialisasi!')
print('=' * 60)


## 2. Memuat Dataset Flipkart dari Kaggle
Dataset yang digunakan adalah **Flipkart E-Commerce Product Reviews** (PromptCloudHQ, Kaggle)
yang juga disebutkan dalam jurnal asli pada bagian *Availability of Data and Materials*.
- **Sumber:** http://www.kaggle.com/PromptCloudHQ/flipkart-products
- **Ukuran:** ~38 MB, 20.000 baris, 15 kolom


In [ ]:
DATASET_PATH = 'data/flipkart_com-ecommerce_sample.csv'

df_raw = spark.read \
    .option('header', 'true') \
    .option('multiLine', 'true') \
    .option('escape', '"') \
    .csv(DATASET_PATH)

total_rows = df_raw.count()
print(f'Total baris dimuat : {total_rows:,}')
print(f'Jumlah kolom       : {len(df_raw.columns)}')
print('Kolom-kolom tersedia:')
print(df_raw.columns)
df_raw.select('product_name', 'retail_price', 'discounted_price', 'overall_rating').show(5, truncate=50)


## 3. Praproses Data & Pembersihan (Spark SQL Transformations)
Tahap ini meliputi:
- Ekstraksi kategori utama (`root_category`) dari kolom JSON string `product_category_tree`
- Konversi dan normalisasi harga numerik
- Pembentukan variabel target `label` (1 = sukses, 0 = gagal)
- Pembersihan teks ulasan/deskripsi produk


In [ ]:
from pyspark.sql.functions import (
    col, when, regexp_replace, regexp_extract,
    lower, coalesce, lit, length, trim
)
from pyspark.sql.types import DoubleType

df_clean = df_raw.withColumn(
    'root_category',
    regexp_extract(col('product_category_tree'), r'\["([^">]+)', 1)
)
df_clean = df_clean.withColumn(
    'root_category',
    when(col('root_category') == '', lit('Other')).otherwise(trim(col('root_category')))
)

df_clean = df_clean.withColumn('retail_price_num', col('retail_price').cast(DoubleType()))
df_clean = df_clean.withColumn('discounted_price_num', col('discounted_price').cast(DoubleType()))
df_clean = df_clean.fillna({'retail_price_num': 1000.0, 'discounted_price_num': 1000.0})
df_clean = df_clean.withColumn(
    'discount_ratio',
    when(col('retail_price_num') > 0,
         (col('retail_price_num') - col('discounted_price_num')) / col('retail_price_num')
    ).otherwise(lit(0.0))
)

df_clean = df_clean.withColumn('rating_numeric', col('overall_rating').cast(DoubleType()))
success_proxy = (col('discount_ratio') >= 0.35) & (length(col('description')) > 100)
df_clean = df_clean.withColumn(
    'label',
    when(col('rating_numeric') >= 4.0, lit(1.0))
    .when((col('rating_numeric') < 4.0) & col('rating_numeric').isNotNull(), lit(0.0))
    .when(success_proxy, lit(1.0)).otherwise(lit(0.0))
)
df_clean = df_clean.withColumn(
    'cleaned_text',
    regexp_replace(lower(coalesce(col('description'), col('product_name'), lit(''))), r'[^a-zA-Z0-9\s]', ' ')
)
df_clean.cache()
print(f'Dataset bersih: {df_clean.count():,} baris')
print('Distribusi Label Target (0=Gagal, 1=Sukses):')
df_clean.groupBy('label').count().orderBy('label').show()


## 4. Seleksi Fitur: Pipeline TF-IDF & VectorAssembler
Pipeline NLP Spark MLlib:
1. **RegexTokenizer** — Tokenisasi teks ulasan produk
2. **StopWordsRemover** — Penghapusan kata-kata umum (stopwords)
3. **HashingTF** — Term Frequency (500 fitur, hashing trick)
4. **IDF** — Inverse Document Frequency untuk pembobotan kata signifikan
5. **StringIndexer** — Encoding kategori produk ke indeks numerik
6. **VectorAssembler** — Penggabungan fitur TF-IDF + numerik


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover, HashingTF, IDF,
    StringIndexer, VectorAssembler
)

tokenizer  = RegexTokenizer(inputCol='cleaned_text', outputCol='tokens', pattern=r'\s+', minTokenLength=3)
remover    = StopWordsRemover(inputCol='tokens', outputCol='filtered_tokens')
hashing_tf = HashingTF(inputCol='filtered_tokens', outputCol='raw_tf', numFeatures=500)
idf        = IDF(inputCol='raw_tf', outputCol='tfidf_features')
indexer    = StringIndexer(inputCol='root_category', outputCol='category_idx', handleInvalid='keep')
assembler  = VectorAssembler(
    inputCols=['tfidf_features', 'retail_price_num', 'discount_ratio', 'category_idx'],
    outputCol='features', handleInvalid='skip'
)
feature_pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf, indexer, assembler])
print('Feature pipeline (NLP + numerik) berhasil dikonfigurasi.')
for i, stage in enumerate(feature_pipeline.getStages()):
    print(f'  {i+1}. {type(stage).__name__}')


## 5. Implementasi Replikasi DMRDF — Filter Redundansi Terdistribusi
Ini merupakan inti dari replikasi paper Narayanan et al. (2020).

**Operasi RDD yang digunakan:**
- `map(β)` — Membentuk pasangan `(hash_key, (record, ig_score, text_len))`
- `filter(β)` — Eliminasi record dengan IG rendah (tidak signifikan)
- `reduceByKey(β)` — Eliminasi duplikasi terdistribusi; pertahankan hanya kemunculan pertama

**Formula Feature Information Gain (ΔG) yang diimplementasikan:**

```
Delta_IG_approx = min(1.0, (unique_tokens / total_tokens) * log2(n_unique + 2) / 4)
```

Sesuai Persamaan (4) dari jurnal: ulasan dengan ΔG = 0 dieliminasi sebagai tidak signifikan.


In [ ]:
import math, hashlib

def dmrdf_map(row):
    text = str(row['cleaned_text']) if row['cleaned_text'] else ''
    category = str(row['root_category']) if row['root_category'] else 'Other'
    tokens = text.split()
    unique_tokens = set(tokens)
    n_tokens, n_unique = len(tokens), len(unique_tokens)
    approx_ig = min(1.0, (n_unique/(n_tokens+1e-5)) * math.log2(n_unique+2) / 4.0) if n_tokens > 0 else 0.0
    key = hashlib.md5(f'{category[:50].lower()}:::{text[:150].lower()}'.encode()).hexdigest()
    return (key, (row, approx_ig, len(text)))

print('Menjalankan DMRDF RDD Pipeline...')
t0 = time.time()
raw_count = df_clean.count()

# Map -> Filter -> ReduceByKey
mapped_rdd       = df_clean.rdd.map(dmrdf_map)
filtered_rdd     = mapped_rdd.filter(lambda p: p[1][1] >= 0.2 and p[1][2] >= 30)
deduplicated_rdd = filtered_rdd.reduceByKey(lambda a, b: a)
clean_rdd        = deduplicated_rdd.map(lambda p: p[1][0])

df_filtered = spark.createDataFrame(clean_rdd, schema=df_clean.schema)
df_filtered.cache()
filtered_count = df_filtered.count()
exec_t = time.time() - t0

removed = raw_count - filtered_count
print(f'Data Awal           : {raw_count:,} baris')
print(f'Setelah DMRDF       : {filtered_count:,} baris')
print(f'Redundansi Dihapus  : {removed:,} baris ({removed/raw_count*100:.2f}%)')
print(f'Waktu RDD Filter    : {exec_t:.3f} detik')


## 6. Pelatihan & Evaluasi Model Klasifikasi (Spark MLlib)
Sesuai paper: proporsi **75% training : 25% testing**, random seed tetap = 42.

Model yang diuji: Logistic Regression, SVM (LinearSVC), Decision Tree Classifier


In [ ]:
from pyspark.ml.classification import LogisticRegression, LinearSVC, DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

def evaluate(df, label, seed=42):
    pm = feature_pipeline.fit(df)
    assembled = pm.transform(df)
    tr, te = assembled.randomSplit([0.75, 0.25], seed=seed)
    tr.cache(); te.cache()
    clfs = [
        ('Logistic Regression', LogisticRegression(featuresCol='features', labelCol='label', maxIter=20, regParam=0.1)),
        ('SVM (LinearSVC)',     LinearSVC(featuresCol='features', labelCol='label', maxIter=20, regParam=0.1)),
        ('Decision Tree',      DecisionTreeClassifier(featuresCol='features', labelCol='label', maxDepth=5, maxBins=300)),
    ]
    acc_ev  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
    prec_ev = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision')
    rec_ev  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall')
    f1_ev   = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
    auc_ev  = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
    rows = []
    print(f'\n=== {label} | Train={tr.count():,} | Test={te.count():,} ===')
    for name, clf in clfs:
        t1 = time.time(); m = clf.fit(tr); tt = time.time()-t1
        t2 = time.time(); p = m.transform(te); p.cache(); p.count(); ts = time.time()-t2
        acc=acc_ev.evaluate(p); prec=prec_ev.evaluate(p); rec=rec_ev.evaluate(p); f1=f1_ev.evaluate(p)
        try: auc=auc_ev.evaluate(p)
        except: auc=0.0
        row = dict(model=name, dataset=label, accuracy=round(acc*100,2),
                   precision=round(prec,4), recall=round(rec,4), f1=round(f1,4),
                   auc_roc=round(auc,4), train_s=round(tt,3), test_s=round(ts,3))
        rows.append(row)
        print(f'  {name:<25} PA={acc*100:.2f}% P={prec:.4f} R={rec:.4f} F1={f1:.4f} T={tt+ts:.2f}s')
    return rows

res_base  = evaluate(df_clean,    label='Tanpa DMRDF')
res_dmrdf = evaluate(df_filtered, label='Dengan DMRDF')


## 7. Visualisasi Perbandingan Kinerja Model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_metrics = pd.DataFrame(res_base + res_dmrdf)
df_metrics['total_s'] = df_metrics['train_s'] + df_metrics['test_s']
short = {'Logistic Regression':'LR', 'SVM (LinearSVC)':'SVM', 'Decision Tree':'DT'}
df_metrics['model_s'] = df_metrics['model'].map(short)
print(df_metrics[['dataset','model','accuracy','precision','recall','f1','auc_roc','train_s','test_s']].to_string(index=False))

palette = {'Tanpa DMRDF':'#e74c3c', 'Dengan DMRDF':'#27ae60'}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Replikasi DMRDF — Flipkart E-Commerce | Narayanan et al. (2020)\nIF25-40402 ITERA | PySpark 3.5.1', fontweight='bold')

for ax, (metric, title, ylim) in zip(axes, [
    ('accuracy', 'Prediction Accuracy (PA %)', (85,100)),
    ('f1',       'F1-Score (Weighted)',         (0.85,1.0)),
    ('total_s',  'Waktu Eksekusi Total (s)',     None)
]):
    sns.barplot(data=df_metrics, x='model_s', y=metric, hue='dataset', palette=palette, ax=ax)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(''); ax.legend(title='', fontsize=9)
    if ylim: ax.set_ylim(ylim)
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(f'{h:.2f}', (p.get_x()+p.get_width()/2, h), ha='center', va='bottom', fontsize=8, xytext=(0,2), textcoords='offset points')

plt.tight_layout()
plt.savefig('results/benchmark_plot.png', dpi=200, bbox_inches='tight')
plt.show()
df_metrics.to_csv('results/metrics_comparison.csv', index=False)
print('Grafik & CSV disimpan ke folder results/')
spark.stop()
print('SparkSession dihentikan.')
